In [52]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



df_ML = pd.read_csv("adult_clean.csv")
df_ML.head(10)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,class
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
2,23,Private,122272,Bachelors,13,Never-married,Adm-clerical,Own-child,White,Female,0,0,30,United-States,<=50K
3,32,Private,205019,Assoc-acdm,12,Never-married,Sales,Not-in-family,Black,Male,0,0,50,United-States,<=50K
4,25,Self-emp-not-inc,176756,HS-grad,9,Never-married,Farming-fishing,Own-child,White,Male,0,0,35,United-States,<=50K
5,32,Private,186824,HS-grad,9,Never-married,Machine-op-inspct,Unmarried,White,Male,0,0,40,United-States,<=50K
6,19,Private,168294,HS-grad,9,Never-married,Craft-repair,Own-child,White,Male,0,0,40,United-States,<=50K
7,23,Local-gov,190709,Assoc-acdm,12,Never-married,Protective-serv,Not-in-family,White,Male,0,0,52,United-States,<=50K
8,20,Private,266015,Some-college,10,Never-married,Sales,Own-child,Black,Male,0,0,44,United-States,<=50K
9,48,Private,242406,11th,7,Never-married,Machine-op-inspct,Unmarried,White,Male,0,0,40,Puerto-Rico,<=50K


In [25]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline



X = df_ML.drop(columns=["class", "fnlwgt", "education"])
y = df_ML["class"].map({
    "<=50K": 0,
    ">50K": 1
})
y.value_counts()


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)




# Preproccessing

In [41]:
scaler1 = MinMaxScaler()
capital_features = ['capital_gain','capital_loss']

# ces trois colonnes ne me semble pas avoir besoin de scaling
scaler2 = StandardScaler()
numerical_features = ["age","education_num","hours_per_week"]


nominales_features = ['workclass','marital_status','occupation','relationship','race','sex','native_country']
onehot_encoder = OneHotEncoder(
    sparse_output=False,      # Retourne array dense avec des 0 (pas sparse matrix)
    handle_unknown='ignore'   # Ignore les catégories inconnues en test
)





In [43]:
preprocessor = ColumnTransformer(
    transformers=[
        ("capital", scaler1, capital_features),
        ("nominales", onehot_encoder, nominales_features),
        ("numeriques", scaler2, numerical_features)
    ],
    remainder="passthrough" # garder les colonnes non modifiées
)

# Logistic Regression


In [44]:
from sklearn.linear_model import LogisticRegression
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

# max_iter=2000 pour permettre la convergence du solver lbfgs

# recherche des meilleurs paramètres sur le modèles afin de l'optimiser, je choisis C en paramètres de validation croisée afin de trouver le niveau de régularisation qui généralise le mieux sur mes données et weight = balanced pour compenser l'inégalité de répartition des classes initiale. Je reste sur le f1 score ici pour ne pas privilégier les FN ou les FP étant donné la problèmatique initiale.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1"
)

grid.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=2000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.1, 1, ...], 'model__class_weight': [None, 'balanced']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of 

In [46]:
print(grid.best_params_)
print(grid.best_score_)

{'model__C': 10, 'model__class_weight': 'balanced'}
0.6811388197783459


In [48]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Meilleur modèle trouvé par GridSearchCV
best_model = grid.best_estimator_

# Prédictions sur le train
y_train_pred = best_model.predict(X_train)

# Prédictions sur le test
y_test_pred = best_model.predict(X_test)

print("TRAIN")
print("Accuracy :", accuracy_score(y_train, y_train_pred))
print("Precision :", precision_score(y_train, y_train_pred))
print("Recall :", recall_score(y_train, y_train_pred))
print("F1 :", f1_score(y_train, y_train_pred))

print("\nTEST")
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision :", precision_score(y_test, y_test_pred))
print("Recall :", recall_score(y_test, y_test_pred))
print("F1 :", f1_score(y_test, y_test_pred))

TRAIN
Accuracy : 0.8092229774756764
Precision : 0.5724005324656116
Recall : 0.8491497531541415
F1 : 0.6838361973759773

TEST
Accuracy : 0.8057362192131358
Precision : 0.5665598601806
Recall : 0.8534444931987714
F1 : 0.6810224089635855


# Les performances obtenues sur les jeux d'entraînement et de test sont très proches, ce qui ne met pas en évidence de surapprentissage. Le modèle présente un rappel élevé sur la classe >50K (0,853 sur le test), indiquant qu'il identifie une grande partie des individus appartenant réellement à cette classe. En revanche, sa précision est plus faible (0,567), traduisant un nombre plus important de faux positifs. Ce déséquilibre entre précision et rappel limite le F1-score à environ 0,68.

In [57]:
y_prob = grid.best_estimator_.predict_proba(X_test)[:, 1]
print(y_prob)
y_pred_05 = (y_prob >= 0.5).astype(int)
y_pred_05[:10]

[0.31470213 0.98641577 0.27336663 ... 0.00936699 0.02428678 0.712284  ]


array([0, 1, 0, 0, 0, 0, 0, 0, 1, 0])

In [62]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Probabilité d'appartenir à la classe >50K
y_prob = grid.best_estimator_.predict_proba(X_test)[:, 1]

for seuil in [0.4, 0.5, 0.6]:

    # Transformation des probabilités en classes 0 ou 1 selon le seuil fixé 
    y_pred_seuil = (y_prob >= seuil).astype(int)

    print(f"Seuil : {seuil}")
    print("Accuracy  :", accuracy_score(y_test, y_pred_seuil))
    print("Precision :", precision_score(y_test, y_pred_seuil))
    print("Recall    :", recall_score(y_test, y_pred_seuil))
    print("F1        :", f1_score(y_test, y_pred_seuil))
    print("ROC_AUC   :", roc_auc_score(y_test, y_pred_seuil))
    print()



Seuil : 0.4
Accuracy  : 0.771510822049259
Precision : 0.5169491525423728
Recall    : 0.910048266783677
F1        : 0.6593546336035606
ROC_AUC   : 0.8185452601524019

Seuil : 0.5
Accuracy  : 0.8057362192131358
Precision : 0.5665598601806
Recall    : 0.8534444931987714
F1        : 0.6810224089635855
ROC_AUC   : 0.8219335142050194

Seuil : 0.6
Accuracy  : 0.8289796353555816
Precision : 0.6181309065453273
Recall    : 0.7749012724879333
F1        : 0.6876947040498442
ROC_AUC   : 0.8106196503284736



# J'ai étudié l'effet d'une modification du seuil de décision sur les performances du modèle. Cependant, ma problématique métier ne donne pas de priorité particulière aux faux positifs ou aux faux négatifs. J'ai donc conservé le seuil de décision à 0,5 et évalué le modèle avec plusieurs métriques, notamment l'accuracy, la précision, le rappel et le F1-score.

In [ ]:
from sklearn.metrics import confusion_matrix


# Calcul de la matrice de confusion
cm = confusion_matrix(y_test, y_test_pred)

# Récupération des 4 valeurs
tn, fp, fn, tp = cm.ravel()

# Labels personnalisés
labels = [
    [f"TN\n{tn}", f"FP\n{fp}"],
    [f"FN\n{fn}", f"TP\n{tp}"]
]

# Création du graphique
plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    linewidths=1,
    linecolor="white",
    xticklabels=["≤50K", ">50K"],
    yticklabels=["≤50K", ">50K"],
    annot_kws={"size": 14, "weight": "bold"}
)

plt.title(
    "Matrice de confusion — Régression logistique",
    fontsize=16,
    fontweight="bold",
    pad=15
)

plt.xlabel("Classe prédite", fontsize=12)
plt.ylabel("Classe réelle", fontsize=12)

plt.tight_layout()
plt.show()


# permutation va me permettre de voir ici lorsque je fais varier de manière aléatoire une caratéristique les résultats du modèles changent significativement. 

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    grid.best_estimator_,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=10,
    random_state=42
)


In [60]:
import pandas as pd

importance_df = pd.DataFrame({
    "variable": X_test.columns,
    "importance": result.importances_mean,
    "ecart_type": result.importances_std
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print(importance_df)

          variable  importance  ecart_type
3   marital_status    0.135186    0.004878
2    education_num    0.048107    0.003311
8     capital_gain    0.035286    0.001677
4       occupation    0.028402    0.002351
5     relationship    0.013921    0.002319
0              age    0.008665    0.001800
10  hours_per_week    0.007341    0.001863
9     capital_loss    0.006757    0.000444
1        workclass    0.006616    0.001053
7              sex    0.004121    0.001579
11  native_country    0.000774    0.001506
6             race    0.000398    0.001058


## Modèle 2 : Random Forest


In [63]:
from sklearn.ensemble import RandomForestClassifier


pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

In [64]:
param_grid_rf = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

# ici il y aura 2*3*2*2 * 5 plis de validation, soit 120 entrainements différents

In [65]:
from sklearn.model_selection import GridSearchCV

grid_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=5,
    scoring="f1",
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [None, 10, ...], 'classifier__min_samples_leaf': [1, 2], 'classifier__min_samples_split': [2, 5], 'classifier__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Su

In [66]:
print("Meilleurs paramètres :", grid_rf.best_params_)
print("Meilleur F1 CV :", grid_rf.best_score_)

Meilleurs paramètres : {'classifier__max_depth': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Meilleur F1 CV : 0.6859313104293419


In [67]:
y_pred_rf = grid_rf.best_estimator_.predict(X_test)

print("Accuracy  :", accuracy_score(y_test, y_pred_rf))
print("Precision :", precision_score(y_test, y_pred_rf))
print("Recall    :", recall_score(y_test, y_pred_rf))
print("F1-score  :", f1_score(y_test, y_pred_rf))

Accuracy  : 0.8657639407186267
Precision : 0.7789934354485777
Recall    : 0.6248354541465555
F1-score  : 0.6934502069637205


In [69]:
import numpy as np

foret = grid_rf.best_estimator_.named_steps["classifier"]
importances = foret.feature_importances_

indices = np.argsort(importances)[::-1]
print(indices)

[ 0 86 12 85 32 87  1 14 20 26 33 37 35 24 11 36 44 43  7  5  6 21 19 28
  2 29 17 42 23 22 82 30  3 40 69  9  8 27 31 15 39 34 16 73 46 38 62 55
 53 13 41 49 47 65 67 78 76 74 83 56 66 79 45 25 75 48 63 52 54 57 50 64
 58 84 81 51 72 70 10 68 60 61 77 80 18 59 71  4]
